# CardioAgent-Ops — QLoRA Qwen2.5-7B

Notebook d'exécution **Google Colab** (GPU T4 16 Go ou L4 24 Go).

> ⚠️ Projet académique. Le modèle n'est pas un dispositif médical et ne doit pas être utilisé
> pour un diagnostic.

1. Cloner le dépôt dans `/content` (ou monter Google Drive).
2. Exécuter les cellules dans l'ordre.
3. Télécharger `llm_finetuning/outputs/`, `llm_finetuning/export/` et `llm_finetuning/mlruns.db`.

In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1), 'Go')

In [ ]:
# Adapter le chemin si le dépôt est ailleurs.
%cd /content/cadrioagent-ops
!pip install -U pip -q
!pip install -r llm_finetuning/requirements-llm.txt -q

In [ ]:
!python llm_finetuning/data/build_dataset.py
!python llm_finetuning/data/build_dataset.py --check
!python -m pytest llm_finetuning/tests -q

In [ ]:
!python -m llm_finetuning.src.train --config llm_finetuning/configs/qlora_qwen25_7b.yaml --dry-run

In [ ]:
# Entraînement complet (2 époques). Réduire max_steps pour un smoke test.
!python -m llm_finetuning.src.train --config llm_finetuning/configs/qlora_qwen25_7b.yaml

In [ ]:
!python -m llm_finetuning.src.evaluate \
  --config llm_finetuning/configs/qlora_qwen25_7b.yaml \
  --adapter llm_finetuning/outputs/qwen25-7b-cardio/adapter \
  --compare-base

In [ ]:
!git clone https://github.com/ggerganov/llama.cpp llm_finetuning/export/llama.cpp
!cmake -S llm_finetuning/export/llama.cpp -B llm_finetuning/export/llama.cpp/build
!cmake --build llm_finetuning/export/llama.cpp/build --config Release -j
!python -m llm_finetuning.src.export_gguf \
  --config llm_finetuning/configs/qlora_qwen25_7b.yaml \
  --adapter llm_finetuning/outputs/qwen25-7b-cardio/adapter \
  --output-dir llm_finetuning/export --quantization q4_k_m

## Récupération des artefacts

- Le GGUF et le `Modelfile` sont dans `llm_finetuning/export/`.
- L'adaptateur est dans `llm_finetuning/outputs/qwen25-7b-cardio/adapter/`.
- La base MLflow locale est `llm_finetuning/mlruns.db` (à ouvrir avec
  `mlflow ui --backend-store-uri sqlite:///llm_finetuning/mlruns.db`).

Sur la machine locale :

```bash
ollama create cardio-llm -f llm_finetuning/export/Modelfile
ollama run cardio-llm
```

> L'intégration Docker (`llm-server`) se fera après validation du fine-tuning.